this is a text.


In [2]:
import pandas as pd
import duckdb

In [3]:
events = pd.read_csv("./data/events.csv")
sessions = pd.read_csv("./data/sessions.csv")

con = duckdb.connect()

In [23]:
df_events = con.execute("select * from events").df()
display(df_events.head(10))
display(df_events.info())
display(df_events.duplicated().sum())

,unique_session_id,event_name,plan_destination,plan_quota
0,1724145563-26F74962FC3C4ABA829693247D77E175,click_to_order_1,AD,NaN
1,1722660565-01C45A959DF448EBBF654E67AF5D5603,click_to_order_1,AD,NaN
2,1723054917-19C075C944554A98B1215281A9D9C25D,click_to_order_1,AD,NaN
3,1723420916-BFC61252529C4501AB422CE63919463D,click_to_order_1,AD,NaN
4,1722759892-30196b203479ee63490e93890e29b308,click_to_order_1,AD,NaN
5,1723473315-9241d7ddf79c43237794a4d88258e5a6,click_to_order_1,AD,NaN
6,1723259409-1E2BB97B31A4416EB9C311CE1A242855,click_to_order_1,AD,NaN
7,1723378402-EF02C278A4CE4A109626D77D3F8B3EAD,click_to_order_1,AD,NaN
8,1724785897-C369D58E4B7C49729514FA947F6AAE85,click_to_order_1,AD,NaN
9,1723457245-b4dea075bf6bb1a9d1f4117e5eff1852,click_to_order_1,AD,NaN


<class 'pandas.DataFrame'>
RangeIndex: 252450 entries, 0 to 252449
Data columns (total 4 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   unique_session_id  252450 non-null  str    
 1   event_name         252450 non-null  str    
 2   plan_destination   252449 non-null  str    
 3   plan_quota         82013 non-null   float64
dtypes: float64(1), str(3)
memory usage: 7.7 MB


None

np.int64(0)

In [24]:
df_sessions = con.execute("select * from sessions").df()
display(df_sessions.head(10))
display(df_sessions.info())
display(df_sessions.duplicated().sum())

,session_date,platform,geo_country,esim_compatible,unique_session_id,transaction_id,transaction_plan_destination,transaction_plan_quota,total_billings_in_usd
0,2024-08-01,ANDROID,Tunisia,true,1722530830-9c47c790289045bf156fe4e25a60e3a7,NaN,NaN,NaN,0.00
1,2024-08-01,ANDROID,United States,true,1722545323-af34b43d2d26d90286fb8fd935e6404e,NaN,NaN,NaN,0.00
2,2024-08-01,ANDROID,Bulgaria,true,1722460762-432eeceae7d473d14902b81c473136cc,NaN,NaN,NaN,0.00
3,2024-08-01,ANDROID,United States,true,1722491702-b9f93edf87528a04033983dd3a16fbd2,NaN,NaN,NaN,0.00
4,2024-08-01,ANDROID,United States,true,1722540669-d87cb7dec85cc212150325d959eb35cd,582be001-336a-4d32-bb88-90a828120b71,ID,3.0,7.64
5,2024-08-01,ANDROID,Canada,true,1722529123-3a5f014b6c492e26c94672c00375a3d8,NaN,NaN,NaN,0.00
6,2024-08-01,ANDROID,Canada,true,1722491001-16a5fcf87de736d5abd00e6061faffeb,7e4d7484-2199-43b5-9e65-8a0705c8a3ec,JP,3.0,6.99
7,2024-08-01,ANDROID,United States,true,1722530078-169604f35b85528b7d278666de737bb0,NaN,NaN,NaN,0.00
8,2024-08-01,ANDROID,United Kingdom,true,1722538138-0e5cd2f6e4e524950bd6e0fc72924b88,NaN,NaN,NaN,0.00
9,2024-08-01,ANDROID,United Kingdom,true,1722501596-6a2142c17fbc28fbad702d97dda3e49e,NaN,NaN,NaN,0.00


<class 'pandas.DataFrame'>
RangeIndex: 274695 entries, 0 to 274694
Data columns (total 9 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   session_date                  274695 non-null  str    
 1   platform                      274695 non-null  str    
 2   geo_country                   274695 non-null  str    
 3   esim_compatible               274695 non-null  str    
 4   unique_session_id             274695 non-null  str    
 5   transaction_id                25133 non-null   str    
 6   transaction_plan_destination  25133 non-null   str    
 7   transaction_plan_quota        25133 non-null   float64
 8   total_billings_in_usd         274695 non-null  float64
dtypes: float64(2), str(7)
memory usage: 18.9 MB


None

np.int64(0)

Checking for unique_session_id duplicates, variations and data meaning in the tables.

Sessions table holds billing and transactiosn information.
Important finding for funnel:

- Session '1724348149-aad4194b0654d8b75d2f6c1b43100ff3' shows important thing - it means the user selected or clicked a specific plan option before checkout, but if there is no transaction_id - there is no purchase. So the user showed purchase intent but abandoned before completing the purchase.

Likely event logic:

- Session started -> row apears in sessions
- eSIM compatible -> true/false (iOS always true, only Android has unsuported models)
- Plan interest -> event `click_to_order_1`
- Plan selected / deeper checkout intent -> event `click_to_order_2`with `plan_quota`
- Purchase completed -> `transaction_id IS NOT NULL` and `total_billings_in_usd > 0`
- Purchase NOT completed -> `transaction_id IS NULL` and/or `total_billings_in_usd < 1`


In [50]:
query = """
SELECT
    unique_session_id, count(*)
FROM sessions
GROUP BY unique_session_id
HAVING COUNT(*) > 1
"""
display(con.execute(query).df())

query = """
SELECT
    *
FROM sessions
where unique_session_id in ('1724304552-a3d87378f11e2c8439c5c1868b4ed401',
                            '1724314216-84166c902629624367b7eb1e160478de',
                            '1724348149-aad4194b0654d8b75d2f6c1b43100ff3')
order by unique_session_id, session_date
"""
display(con.execute(query).df())

query = """
SELECT
    *
FROM events
where unique_session_id in ('1724304552-a3d87378f11e2c8439c5c1868b4ed401',
                            '1724314216-84166c902629624367b7eb1e160478de', 
                            '1724348149-aad4194b0654d8b75d2f6c1b43100ff3')
order by unique_session_id, event_name
"""
display(con.execute(query).df())

,unique_session_id,count_star()
0,1724335297-a11f09f6187c5efe3416913354059427,2
1,1724273986-68eca0be8b7c887894ae2621b0fd153a,2
2,1724286722-a2b177ed118e884bcc3e5fb9663ec6ee,2
3,1722583517-f65f224c54ccd8f70b6385495c257df3,2
4,1724336904-59b73388049c34334e4fedf3ac2920f9,2
...,...,...
30831,1724205933-259BA76EACA749C79B8D28B4436679A6,2
30832,1724223717-F2DB98850005443EAADA2053F89D0665,2
30833,1724207566-BFAAC47741414E6B863724E86093570F,2
30834,1724279898-458812b04ae27b41c4e6270eaa827a85,2


,session_date,platform,geo_country,esim_compatible,unique_session_id,transaction_id,transaction_plan_destination,transaction_plan_quota,total_billings_in_usd
0,2024-08-22,ANDROID,Bulgaria,true,1724304552-a3d87378f11e2c8439c5c1868b4ed401,747cec06-a607-4573-875e-bd85a0d71114,BG,5.0,6.99
1,2024-08-22,ANDROID,Bulgaria,true,1724304552-a3d87378f11e2c8439c5c1868b4ed401,NaN,NaN,NaN,0.00
2,2024-08-22,ANDROID,Sweden,true,1724314216-84166c902629624367b7eb1e160478de,3005c154-c741-42ec-99de-23ac0f2f13e9,US,5.0,13.99
3,2024-08-22,ANDROID,Sweden,true,1724314216-84166c902629624367b7eb1e160478de,NaN,NaN,NaN,0.00
4,2024-08-22,ANDROID,United Kingdom,true,1724348149-aad4194b0654d8b75d2f6c1b43100ff3,NaN,NaN,NaN,0.00
5,2024-08-24,ANDROID,United Kingdom,true,1724348149-aad4194b0654d8b75d2f6c1b43100ff3,NaN,NaN,NaN,0.00


,unique_session_id,event_name,plan_destination,plan_quota
0,1724304552-a3d87378f11e2c8439c5c1868b4ed401,click_to_order_1,BG,NaN
1,1724304552-a3d87378f11e2c8439c5c1868b4ed401,click_to_order_2,BG,5.0
2,1724314216-84166c902629624367b7eb1e160478de,click_to_order_1,US,NaN
3,1724314216-84166c902629624367b7eb1e160478de,click_to_order_2,US,5.0
4,1724348149-aad4194b0654d8b75d2f6c1b43100ff3,click_to_order_1,DZ,NaN
5,1724348149-aad4194b0654d8b75d2f6c1b43100ff3,click_to_order_2,DZ,1.0


Check conversions rate


In [ ]:
query = """
SELECT
    platform,
    esim_compatible,
    COUNT(DISTINCT unique_session_id) AS total_sessions,
    COUNT(DISTINCT CASE 
        WHEN transaction_id IS NOT NULL THEN unique_session_id 
    END) AS purchasing_sessions,
    ROUND(
        COUNT(DISTINCT CASE 
            WHEN transaction_id IS NOT NULL THEN unique_session_id 
        END) * 100.0 
        / COUNT(DISTINCT unique_session_id),
        2
    ) AS conversion_rate_pct,
    ROUND(SUM(total_billings_in_usd), 2) AS revenue_usd
FROM sessions
GROUP BY platform, esim_compatible
ORDER BY conversion_rate_pct DESC
"""

platform_cr = con.execute(query).df()
display(platform_cr)

#  Duplicated unique_session_id exists
print(f"total_sessions duplicated: {274695 - (128872+85603+27073)}")
print(f"NOT purchasing_sessions: {274695 - (16088+8258+110)}")

,platform,esim_compatible,total_sessions,purchasing_sessions,conversion_rate_pct,revenue_usd
0,IOS,not_applicable,128872,16088,12.48,240442.20
1,ANDROID,true,85603,8258,9.65,116945.46
2,ANDROID,false,27073,110,0.41,1238.19


total_sessions duplicated: 33147
NOT purchasing_sessions: 250239


In [ ]:
query = """
SELECT
    *
FROM sessions
where unique_session_id in ('1724723941-85d9250152d7174971df6119269db913')
order by unique_session_id, session_date
"""
display(con.execute(query).df())

query = """
SELECT
    *
FROM events
where unique_session_id in ('1724723941-85d9250152d7174971df6119269db913')
order by unique_session_id, event_name
"""
display(con.execute(query).df())